# KD-CIFAR10: Knowledge Distillation Ablation Study

**Goal:** Compare Logit-KD vs Feature-KD for compressing ResNet-50 (teacher) → ResNet-18 (student) on CIFAR-10.

**Loss:** `L_total = α · L_kd + (1 - α) · L_ce` (Hinton et al., 2015)

## Notebook Structure

| Section | Content |
|---------|---------|
| 0 | Environment, repo, data, dependencies |
| 2 | Train ResNet-50 teacher |
| **3 (NEW)** | **Multi-seed baselines (R18, R34) + corrected Logit-KD & Feature-KD — 3 seeds each** |
| 4 | Full ablation grid — Experiment 2, single seed |
| 5 | Analysis & visualization |

> **Why Section 3?** The original `FeatureKDLoss` silently ignored the `student_channels` /
> `teacher_channels` constructor arguments and used layer-4 dimensions (512→2048) for every
> projection layer, including layer1 (should be 64→256). The fix uses per-layer channel lookup.
> Section 3 provides corrected Feature-KD results plus 3-seed variance estimates for all configs.

---
## Section 0: Environment & Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in sys.modules
print(f'In Colab: {IN_COLAB}')

import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training on CPU will be very slow.')

In [ ]:
import subprocess

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO = Path('/content/kd-cifar10')
    if not REPO.exists():
        subprocess.run(
            ['git', 'clone', 'https://github.com/umutonuryasar/kd-cifar10.git', str(REPO)],
            check=True
        )
    else:
        subprocess.run(['git', 'pull'], cwd=str(REPO), check=True)

    os.chdir(REPO)
    sys.path.insert(0, str(REPO))
    print(f'CWD: {os.getcwd()}')
else:
    repo_root = Path.cwd()
    while not (repo_root / 'tools' / 'train.py').exists() and repo_root != repo_root.parent:
        repo_root = repo_root.parent
    os.chdir(repo_root)
    sys.path.insert(0, str(repo_root))
    print(f'CWD: {os.getcwd()}')

In [ ]:
if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'tensorboard', 'matplotlib', 'seaborn', 'pandas'],
        check=True
    )
print('Dependencies ready.')

In [ ]:
import torchvision

DATA_DIR = 'data'

def setup_cifar10(root='data'):
    """Ensure CIFAR-10 is in ImageFolder format under root/."""
    root = Path(root)
    train_dir, test_dir = root / 'train', root / 'test'

    if train_dir.exists() and test_dir.exists():
        n_train = sum(1 for _ in train_dir.rglob('*.png'))
        n_test  = sum(1 for _ in test_dir.rglob('*.png'))
        if n_train >= 50000 and n_test >= 10000:
            print(f'Data ready: {n_train} train, {n_test} test.')
            return

    # Option A: unzip from Google Drive
    zip_path = Path('/content/drive/MyDrive/cifar10-imagefolder.zip')
    if zip_path.exists():
        import zipfile, shutil
        print(f'Extracting {zip_path} ...')
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall('/tmp/cifar_extract')
        shutil.move('/tmp/cifar_extract/cifar10-imagefolder', str(root))
        print('Done.')
        return

    # Option B: download via torchvision and convert to ImageFolder
    classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
    print('Downloading CIFAR-10 and converting to ImageFolder format (~60 k images)...')
    for split, is_train in [('train', True), ('test', False)]:
        ds = torchvision.datasets.CIFAR10(root='/tmp/cifar10_raw', train=is_train, download=True)
        for idx, (img, label) in enumerate(ds):
            d = root / split / classes[label]
            d.mkdir(parents=True, exist_ok=True)
            img.save(d / f'{idx:05d}.png')
        print(f'  {split}: {len(ds)} images.')
    print('CIFAR-10 ImageFolder ready.')

setup_cifar10(DATA_DIR)

In [ ]:
SEEDS        = [42, 123, 456]
TEACHER_CKPT = 'runs/teacher_r50_v2/checkpoint_best.pth'
EPOCHS       = 100
BS           = 128
LR           = 0.1
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device : {DEVICE}')
print(f'Seeds  : {SEEDS}')
print(f'Teacher: {TEACHER_CKPT}')

In [ ]:
import subprocess, sys, torch, numpy as np
from pathlib import Path

def run_train(extra_args, out_dir, skip_if_done=True):
    """Launch a training job; skip gracefully if checkpoint already exists."""
    ckpt = Path(out_dir) / 'checkpoint_best.pth'
    if skip_if_done and ckpt.exists():
        print(f'  [skip] {out_dir}')
        return
    cmd = [
        sys.executable, 'tools/train.py',
        '--epochs', str(EPOCHS),
        '--batch-size', str(BS),
        '--lr', str(LR),
        '--device', DEVICE,
        '--data-dir', DATA_DIR,
        '--output-dir', out_dir,
    ] + extra_args
    print(f'  [run ] {out_dir}')
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f'Training failed: {out_dir}')

def load_best_acc(run_dir):
    """Return best validation accuracy from checkpoint_best.pth, or None."""
    p = Path(run_dir) / 'checkpoint_best.pth'
    if not p.exists():
        return None
    ckpt = torch.load(p, map_location='cpu', weights_only=False)
    return ckpt.get('best_acc')

def summarize_seeds(run_dirs):
    """Compute mean/std best-accuracy across a list of run directories."""
    accs = [load_best_acc(d) for d in run_dirs]
    accs = [a for a in accs if a is not None]
    if not accs:
        return dict(mean=None, std=None, per_seed=accs)
    return dict(mean=float(np.mean(accs)), std=float(np.std(accs)), per_seed=accs)

print('Helpers loaded.')

---
## Section 2: Train Teacher (ResNet-50)

CIFAR-specific modification applied to both teacher and student:
- `conv1`: 7×7 stride-2 → 3×3 stride-1  
- `maxpool`: replaced with `nn.Identity()`

Expected accuracy: ~95%. Takes ~40 min on A100, ~15 min on H100.

In [ ]:
run_train(
    extra_args=['--model', 'resnet50', '--kd-type', 'none', '--seed', '42'],
    out_dir='runs/teacher_r50_v2',
)
teacher_acc = load_best_acc('runs/teacher_r50_v2')
print(f'Teacher best acc: {teacher_acc*100:.2f}%' if teacher_acc else 'Not trained yet.')

---
## Section 3 (NEW): Multi-Seed Baselines & Corrected KD Results

### 3.0  Background — Feature-KD Bug Fix

The original `FeatureKDLoss.__init__` accepted `student_channels` and `teacher_channels`
scalar arguments but **silently ignored them**, instead hard-coding layer-4 dimensions
(512 → 2048) for every projection layer:

```python
# BUG: all four projections were built as Conv2d(512, 2048, 1)
# layer1 should be Conv2d(64, 256, 1)  — wrong by 8×
# layer2 should be Conv2d(128, 512, 1) — wrong by 4×
# layer3 should be Conv2d(256, 1024, 1)— wrong by 2×
```

```python
# FIX: per-layer lookup (current code)
STUDENT_CHANNELS = {'layer1': 64,  'layer2': 128,  'layer3': 256,  'layer4': 512}
TEACHER_CHANNELS = {'layer1': 256, 'layer2': 512,  'layer3': 1024, 'layer4': 2048}
```

With the wrong dimensions, PyTorch would raise a shape mismatch at runtime unless the
feature spatial sizes happened to align (they don't for layer1–3). This means all
Experiment 2 Feature-KD runs were effectively broken or projecting garbage.

**This section provides:**
- R18 standalone baseline × 3 seeds (no KD reference)
- R34 standalone baseline × 3 seeds (larger-model reference)
- All Logit-KD configs × 3 seeds (corrected run dirs for variance reporting)
- All Feature-KD configs × 3 seeds with the bug fixed

Output dirs use `runs/v3/` prefix to avoid collisions with Experiment 2.

### 3.1  R18 Standalone Baselines (3 Seeds, No KD)

In [ ]:
print('=== 3.1  ResNet-18 Baselines ===')
for seed in SEEDS:
    run_train(
        extra_args=['--model', 'resnet18', '--kd-type', 'none', '--seed', str(seed)],
        out_dir=f'runs/v3/baseline_r18_s{seed}',
    )

### 3.2  R34 Standalone Baselines (3 Seeds, No KD)

ResNet-34 uses the same BasicBlock channel sizes as ResNet-18 (64/128/256/512)
and the same CIFAR modification (3×3 conv, no maxpool). It serves as a
"larger student" reference: does KD let R18 close the gap to R34?

In [ ]:
print('=== 3.2  ResNet-34 Baselines ===')
for seed in SEEDS:
    run_train(
        extra_args=['--model', 'resnet34', '--kd-type', 'none', '--seed', str(seed)],
        out_dir=f'runs/v3/baseline_r34_s{seed}',
    )

### 3.3  R50 → R18 Logit-KD — All Configs × 3 Seeds

Grid matches Experiment 2:  
- Alpha sweep: α ∈ {0.3, 0.5, 0.7}, T=2  
- Temperature sweep: T ∈ {2, 3, 4}, α=0.5  
(α=0.5 T=2 is shared between the two sweeps.)

In [ ]:
print('=== 3.3  Logit-KD (all configs) × 3 seeds ===')

LOGIT_CONFIGS = [
    ('0.3', '2'),
    ('0.5', '2'),
    ('0.7', '2'),
    ('0.5', '3'),
    ('0.5', '4'),
]

for alpha, temp in LOGIT_CONFIGS:
    print(f'  -- Logit alpha={alpha} T={temp} --')
    for seed in SEEDS:
        run_train(
            extra_args=[
                '--model', 'resnet18',
                '--kd-type', 'logit',
                '--alpha', alpha,
                '--temperature', temp,
                '--teacher-weights', TEACHER_CKPT,
                '--seed', str(seed),
            ],
            out_dir=f'runs/v3/logit_a{alpha}_t{temp}_s{seed}',
        )

### 3.4  R50 → R18 Feature-KD — All Configs × 3 Seeds (Bug Fixed)

The corrected `FeatureKDLoss` now uses per-layer projection dimensions:
- layer1: Conv2d(64, 256, 1)
- layer2: Conv2d(128, 512, 1)
- layer3: Conv2d(256, 1024, 1)
- layer4: Conv2d(512, 2048, 1)

In [ ]:
print('=== 3.4  Feature-KD (corrected) × 3 seeds ===')

FEATURE_ALPHAS = ['0.3', '0.5', '0.7']

for alpha in FEATURE_ALPHAS:
    print(f'  -- Feature alpha={alpha} --')
    for seed in SEEDS:
        run_train(
            extra_args=[
                '--model', 'resnet18',
                '--kd-type', 'feature',
                '--alpha', alpha,
                '--feat-beta', '0.5',
                '--teacher-weights', TEACHER_CKPT,
                '--seed', str(seed),
            ],
            out_dir=f'runs/v3/feature_a{alpha}_s{seed}',
        )

### 3.5  Multi-Seed Results Summary

In [ ]:
import pandas as pd

def fmt_acc(a):
    return f'{a*100:.2f}%' if a is not None else '—'

def fmt_std(s):
    return f'\u00b1{s*100:.2f}%' if s is not None else '—'

rows = []

# Teacher (single reference run)
t_acc = load_best_acc('runs/teacher_r50_v2')
rows.append(dict(
    Config='Teacher (R50)', Type='teacher',
    Mean=fmt_acc(t_acc), Std='—',
    S42=fmt_acc(t_acc), S123='—', S456='—',
))

# Baselines
for arch in ['r18', 'r34']:
    dirs = [f'runs/v3/baseline_{arch}_s{s}' for s in SEEDS]
    stat = summarize_seeds(dirs)
    per  = stat['per_seed']
    rows.append(dict(
        Config=f'Baseline ({arch.upper()})', Type='baseline',
        Mean=fmt_acc(stat['mean']), Std=fmt_std(stat['std']),
        S42=fmt_acc(per[0] if len(per) > 0 else None),
        S123=fmt_acc(per[1] if len(per) > 1 else None),
        S456=fmt_acc(per[2] if len(per) > 2 else None),
    ))

# Logit-KD
for alpha, temp in LOGIT_CONFIGS:
    dirs = [f'runs/v3/logit_a{alpha}_t{temp}_s{s}' for s in SEEDS]
    stat = summarize_seeds(dirs)
    per  = stat['per_seed']
    rows.append(dict(
        Config=f'Logit \u03b1={alpha} T={temp}', Type='logit',
        Mean=fmt_acc(stat['mean']), Std=fmt_std(stat['std']),
        S42=fmt_acc(per[0] if len(per) > 0 else None),
        S123=fmt_acc(per[1] if len(per) > 1 else None),
        S456=fmt_acc(per[2] if len(per) > 2 else None),
    ))

# Feature-KD
for alpha in FEATURE_ALPHAS:
    dirs = [f'runs/v3/feature_a{alpha}_s{s}' for s in SEEDS]
    stat = summarize_seeds(dirs)
    per  = stat['per_seed']
    rows.append(dict(
        Config=f'Feature \u03b1={alpha} (fixed)', Type='feature',
        Mean=fmt_acc(stat['mean']), Std=fmt_std(stat['std']),
        S42=fmt_acc(per[0] if len(per) > 0 else None),
        S123=fmt_acc(per[1] if len(per) > 1 else None),
        S456=fmt_acc(per[2] if len(per) > 2 else None),
    ))

df3 = pd.DataFrame(rows, columns=['Config', 'Type', 'Mean', 'Std', 'S42', 'S123', 'S456'])
df3.columns = ['Config', 'Type', 'Mean Acc', '\u00b1Std', 'Seed 42', 'Seed 123', 'Seed 456']
print(df3.to_string(index=False))
df3

---
## Section 4: Full Ablation Grid — Experiment 2 (Single Seed)

These runs replicate the original Experiment 2 single-seed ablation (seed=42).
For statistically valid numbers with variance, use Section 3 (multi-seed, corrected).

**Grid:**
- Logit-KD alpha sweep: α ∈ {0.3, 0.5, 0.7}, T=2
- Logit-KD temperature sweep: T ∈ {2, 3, 4}, α=0.5 (T=2 shared with alpha sweep)
- Feature-KD alpha sweep: α ∈ {0.3, 0.5, 0.7}, β=0.5

### 4.1  Logit-KD Alpha Sweep (T=2, seed=42)

In [ ]:
print('=== 4.1  Logit-KD alpha sweep ===')
for alpha in ['0.3', '0.5', '0.7']:
    run_train(
        extra_args=[
            '--model', 'resnet18', '--kd-type', 'logit',
            '--alpha', alpha, '--temperature', '2',
            '--teacher-weights', TEACHER_CKPT, '--seed', '42',
        ],
        out_dir=f'runs/logit_a{alpha}_t2',
    )

### 4.2  Logit-KD Temperature Sweep (α=0.5, seed=42)

T=2 already trained in 4.1 (`runs/logit_a0.5_t2`). Running T=3 and T=4 here.

In [ ]:
print('=== 4.2  Logit-KD temperature sweep ===')
for temp in ['3', '4']:
    run_train(
        extra_args=[
            '--model', 'resnet18', '--kd-type', 'logit',
            '--alpha', '0.5', '--temperature', temp,
            '--teacher-weights', TEACHER_CKPT, '--seed', '42',
        ],
        out_dir=f'runs/logit_a0.5_t{temp}',
    )

### 4.3  Feature-KD Alpha Sweep (β=0.5, seed=42)

In [ ]:
print('=== 4.3  Feature-KD alpha sweep ===')
for alpha in ['0.3', '0.5', '0.7']:
    run_train(
        extra_args=[
            '--model', 'resnet18', '--kd-type', 'feature',
            '--alpha', alpha, '--feat-beta', '0.5',
            '--teacher-weights', TEACHER_CKPT, '--seed', '42',
        ],
        out_dir=f'runs/feature_a{alpha}',
    )

### 4.4  Experiment 2 Results Table

In [ ]:
import pandas as pd

t_acc   = load_best_acc('runs/teacher_r50_v2') or 0.9540
b_acc   = (load_best_acc('runs/baseline_v2')
           or load_best_acc('runs/v3/baseline_r18_s42')
           or 0.9497)

EXP2_RUNS = [
    ('Teacher (R50)',   t_acc,                              None,   None),
    ('Baseline (R18)',  b_acc,                              t_acc,  None),
    ('Logit \u03b1=0.3 T=2', load_best_acc('runs/logit_a0.3_t2'),  t_acc, b_acc),
    ('Logit \u03b1=0.5 T=2', load_best_acc('runs/logit_a0.5_t2'),  t_acc, b_acc),
    ('Logit \u03b1=0.7 T=2', load_best_acc('runs/logit_a0.7_t2'),  t_acc, b_acc),
    ('Logit \u03b1=0.5 T=3', load_best_acc('runs/logit_a0.5_t3'),  t_acc, b_acc),
    ('Logit \u03b1=0.5 T=4', load_best_acc('runs/logit_a0.5_t4'),  t_acc, b_acc),
    ('Feature \u03b1=0.3',   load_best_acc('runs/feature_a0.3'),   t_acc, b_acc),
    ('Feature \u03b1=0.5',   load_best_acc('runs/feature_a0.5'),   t_acc, b_acc),
    ('Feature \u03b1=0.7',   load_best_acc('runs/feature_a0.7'),   t_acc, b_acc),
]

rows = []
for name, acc, ref_t, ref_b in EXP2_RUNS:
    rows.append({
        'Config':     name,
        'Best Acc':   fmt_acc(acc),
        '\u0394 Teacher':  f'{(acc - ref_t)*100:+.2f}pp' if (acc and ref_t) else '\u2014',
        '\u0394 Baseline': f'{(acc - ref_b)*100:+.2f}pp' if (acc and ref_b) else '\u2014',
    })

df4 = pd.DataFrame(rows)
print(df4.to_string(index=False))
df4

---
## Section 5: Analysis & Visualization

In [ ]:
# Launch TensorBoard to browse all training runs interactively.
# Filter to Section 3 multi-seed runs:
%load_ext tensorboard
%tensorboard --logdir runs/v3 --port 6006

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    HAS_TB = True
except ImportError:
    HAS_TB = False

def load_val_acc(logdir, tag='val/acc'):
    if not HAS_TB or not Path(logdir).exists():
        return [], []
    ea = EventAccumulator(logdir)
    ea.Reload()
    try:
        events = ea.Scalars(tag)
        return [e.step for e in events], [e.value for e in events]
    except KeyError:
        return [], []

def mean_curve(run_prefix, seeds, suffix='/tb_logs'):
    """Average val-acc curves across seeds; returns (epochs, mean, std)."""
    curves = []
    for s in seeds:
        _, accs = load_val_acc(f'{run_prefix}{s}{suffix}')
        if accs:
            curves.append(accs)
    if not curves:
        return [], [], []
    min_len = min(len(c) for c in curves)
    mat = np.array([c[:min_len] for c in curves])
    return list(range(1, min_len + 1)), mat.mean(0).tolist(), mat.std(0).tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: alpha effect on Logit-KD (T=2)
ax = axes[0]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for (alpha, temp), col in zip([('0.3','2'), ('0.5','2'), ('0.7','2')], colors):
    prefix = f'runs/v3/logit_a{alpha}_t{temp}_s'
    epochs, mean, std = mean_curve(prefix, SEEDS)
    if epochs:
        mean_arr = np.array(mean)
        std_arr  = np.array(std)
        ax.plot(epochs, mean_arr * 100, color=col, label=f'\u03b1={alpha}')
        ax.fill_between(epochs, (mean_arr - std_arr) * 100,
                        (mean_arr + std_arr) * 100, alpha=0.15, color=col)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Accuracy (%)')
ax.set_title('Logit-KD: Alpha Effect (T=2)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.legend()
ax.grid(True, alpha=0.3)

# Right: temperature effect on Logit-KD (alpha=0.5)
ax = axes[1]
for (alpha, temp), col in zip([('0.5','2'), ('0.5','3'), ('0.5','4')], colors):
    prefix = f'runs/v3/logit_a{alpha}_t{temp}_s'
    epochs, mean, std = mean_curve(prefix, SEEDS)
    if epochs:
        mean_arr = np.array(mean)
        std_arr  = np.array(std)
        ax.plot(epochs, mean_arr * 100, color=col, label=f'T={temp}')
        ax.fill_between(epochs, (mean_arr - std_arr) * 100,
                        (mean_arr + std_arr) * 100, alpha=0.15, color=col)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Accuracy (%)')
ax.set_title('Logit-KD: Temperature Effect (\u03b1=0.5)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
Path('notebooks').mkdir(exist_ok=True)
plt.savefig('notebooks/convergence_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side: Experiment 2 (single seed) vs Experiment 3 (multi-seed, corrected)
import pandas as pd

t_acc = load_best_acc('runs/teacher_r50_v2') or 0.9540

compare_rows = []

def exp3_stat(dirs):
    stat = summarize_seeds(dirs)
    m, s = stat['mean'], stat['std']
    if m is None:
        return '—'
    return f'{m*100:.2f}% {fmt_std(s)}'

# Teacher
compare_rows.append(dict(
    Config='Teacher (R50)',
    Exp2_s42=fmt_acc(t_acc),
    Exp3_MeanStd=fmt_acc(t_acc),
))

# R18 baseline
compare_rows.append(dict(
    Config='Baseline (R18)',
    Exp2_s42=fmt_acc(load_best_acc('runs/baseline_v2') or load_best_acc('runs/v3/baseline_r18_s42')),
    Exp3_MeanStd=exp3_stat([f'runs/v3/baseline_r18_s{s}' for s in SEEDS]),
))

# R34 baseline (new in Exp3)
compare_rows.append(dict(
    Config='Baseline (R34)',
    Exp2_s42='(new)',
    Exp3_MeanStd=exp3_stat([f'runs/v3/baseline_r34_s{s}' for s in SEEDS]),
))

# Logit-KD
for alpha, temp in LOGIT_CONFIGS:
    compare_rows.append(dict(
        Config=f'Logit \u03b1={alpha} T={temp}',
        Exp2_s42=fmt_acc(load_best_acc(f'runs/logit_a{alpha}_t{temp}')),
        Exp3_MeanStd=exp3_stat([f'runs/v3/logit_a{alpha}_t{temp}_s{s}' for s in SEEDS]),
    ))

# Feature-KD
for alpha in FEATURE_ALPHAS:
    compare_rows.append(dict(
        Config=f'Feature \u03b1={alpha}',
        Exp2_s42=fmt_acc(load_best_acc(f'runs/feature_a{alpha}')),
        Exp3_MeanStd=exp3_stat([f'runs/v3/feature_a{alpha}_s{s}' for s in SEEDS]),
    ))

df_cmp = pd.DataFrame(compare_rows)
df_cmp.columns = ['Config', 'Exp2 (seed=42)', 'Exp3 Mean \u00b1 Std (3 seeds)']
print(df_cmp.to_string(index=False))
df_cmp

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Bar chart: multi-seed mean accuracy with std error bars
labels, means, stds, colors_bar = [], [], [], []

COLOR_MAP = {'baseline': '#aec7e8', 'logit': '#1f77b4', 'feature': '#ff7f0e'}

for row in rows:  # df3 rows from cell-20
    cfg = row['Config'] if isinstance(row, dict) else None

for _, row in df3.iterrows():
    typ = row['Type']
    if typ == 'teacher':
        continue
    mean_str = row['Mean Acc'].replace('%', '')
    std_str  = row['\u00b1Std'].replace('\u00b1', '').replace('%', '')
    try:
        labels.append(row['Config'])
        means.append(float(mean_str))
        stds.append(float(std_str) if std_str != '\u2014' else 0)
        colors_bar.append(COLOR_MAP.get(typ, '#cccccc'))
    except ValueError:
        pass

if labels:
    fig, ax = plt.subplots(figsize=(14, 5))
    x = range(len(labels))
    bars = ax.bar(x, means, yerr=stds, capsize=4, color=colors_bar, edgecolor='white')
    t_line = (t_acc or 0.954) * 100
    ax.axhline(t_line, color='red', linestyle='--', linewidth=1.5, label=f'Teacher {t_line:.2f}%')
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Best Val Accuracy (%)')
    ax.set_title('Experiment 3: Multi-Seed Results (mean \u00b1 std, 3 seeds)')
    ax.set_ylim([min(means) - 0.5, t_line + 0.3])
    patches = [mpatches.Patch(color=v, label=k) for k, v in COLOR_MAP.items()]
    ax.legend(handles=patches + [ax.get_lines()[0]], fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('notebooks/multiseed_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No completed runs to plot yet.')